# 01. 자산 데이터 입력 및 버킷 분류

이 Notebook에서 하는 일:
1. 보유 자산 CSV 로드 (또는 수동 입력)
2. 자산 유형별 자동 버킷 분류
3. 현재 포트폴리오 현황 시각화
4. 생활비 안전도(몇 개월치?) 계산
5. DB 스냅샷 저장

In [1]:
# 공통 설정 — 매 Notebook 상단에서 실행
import pandas as pd
import numpy as np
import yaml, sqlite3, os
from pathlib import Path
from dotenv import load_dotenv
from datetime import date

load_dotenv()
PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET = CONFIG['portfolio']

print(f"사용자: {CONFIG['user']['name']}")
print(f"월 생활비: {MONTHLY_EXPENSE:,}원")

사용자: 종헌
월 생활비: 2,500,000원


## 1. 자산 CSV 로드

`assets.csv` 파일이 있으면 자동으로 불러옵니다.  
없으면 샘플 데이터(`assets_sample.csv`)로 시작합니다.

> **실제 데이터 입력 방법:**  
> 1. `assets_sample.csv`를 복사해서 `assets.csv`로 저장  
> 2. 엑셀 또는 메모장으로 열어 본인 자산으로 수정  
> 3. 이 셀을 다시 실행

In [2]:
assets_path  = PROJECT_ROOT / 'assets.csv'
sample_path  = PROJECT_ROOT / 'assets_sample.csv'

if assets_path.exists():
    df = pd.read_csv(assets_path, encoding='utf-8-sig')
    print(f'assets.csv 로드 완료 — {len(df)}개 자산')
elif sample_path.exists():
    df = pd.read_csv(sample_path, encoding='utf-8-sig')
    print(f'샘플 데이터 로드 — {len(df)}개 자산')
    print()
    print('실제 자산으로 교체하려면:')
    print('  assets_sample.csv를 복사 → assets.csv 로 저장 후 내용을 수정하세요.')
else:
    print('CSV 파일을 찾을 수 없습니다. 프로젝트 폴더를 확인하세요.')

# current_value가 없으면 quantity * unit_price로 계산
mask = df['current_value'].isna() | (df['current_value'] == 0)
df.loc[mask, 'current_value'] = df.loc[mask, 'quantity'] * df.loc[mask, 'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)

df

샘플 데이터 로드 — 7개 자산

실제 자산으로 교체하려면:
  assets_sample.csv를 복사 → assets.csv 로 저장 후 내용을 수정하세요.


,account_name,asset_name,ticker,asset_type,quantity,unit_price,current_value,purchase_date
0,연금저축(예시),TIGER 미국S&P500,360750.0,equity,50.0,18500.0,1020000,2023-03-15
1,연금저축(예시),KODEX 단기채권,153130.0,bond,100.0,102000.0,10250000,2022-06-01
2,IRP(예시),TDF2035(미래에셋),189400.0,tdf,30.0,15200.0,480000,2023-09-01
3,IRP(예시),KODEX 국고채3년,114820.0,bond,200.0,101500.0,20400000,2021-11-10
4,예금(예시),정기예금(KB),NaN,cash,NaN,NaN,30000000,2024-01-01
5,예금(예시),CMA(삼성),NaN,cash,NaN,NaN,5000000,2024-03-01
6,증권계좌(예시),TIGER 리츠부동산인프라,329200.0,income,80.0,4800.0,400000,2023-07-20


## 2. 버킷 자동 분류

In [3]:
# asset_type → 버킷 번호 매핑
BUCKET_MAP = {
    'cash':   1,   # 버킷 1: 즉시 인출 (예금, CMA, MMF)
    'bond':   2,   # 버킷 2: 중기 안정 (채권)
    'tdf':    2,   # TDF 안정형은 버킷 2로
    'equity': 3,   # 버킷 3: 장기 성장 (주식형 ETF)
    'income': 3,   # 버킷 3: 인컴/리츠 (배당 수익)
}

df['bucket'] = df['asset_type'].str.lower().map(BUCKET_MAP).fillna(0).astype(int)

# 버킷별 합계
bucket_sum = df.groupby('bucket')['current_value'].sum()
total      = df['current_value'].sum()

b1 = bucket_sum.get(1, 0)   # 현금성
b2 = bucket_sum.get(2, 0)   # 채권/TDF
b3 = bucket_sum.get(3, 0)   # 주식/인컴

print('=== 버킷별 자산 현황 ===')
print(f'버킷 1 (현금성): {b1:>15,.0f}원  ({b1/total*100:.1f}%)')
print(f'버킷 2 (채권/TDF): {b2:>13,.0f}원  ({b2/total*100:.1f}%)')
print(f'버킷 3 (성장/인컴): {b3:>12,.0f}원  ({b3/total*100:.1f}%)')
print(f'{"─"*45}')
print(f'총 자산:         {total:>15,.0f}원')

=== 버킷별 자산 현황 ===
버킷 1 (현금성):      35,000,000원  (51.8%)
버킷 2 (채권/TDF):    31,130,000원  (46.1%)
버킷 3 (성장/인컴):    1,420,000원  (2.1%)
─────────────────────────────────────────────
총 자산:              67,550,000원


## 3. 생활비 안전도 계산

In [4]:
months_covered = b1 / MONTHLY_EXPENSE

if months_covered >= 12:
    level = '녹색 (안전)'
    emoji = '🟢'
elif months_covered >= 6:
    level = '황색 (주의)'
    emoji = '🟡'
else:
    level = '적색 (위험)'
    emoji = '🔴'

print('=== 생활비 안전도 ===')
print(f'{emoji} 상태: {level}')
print(f'현금성 자산: {b1:,.0f}원')
print(f'월 생활비:   {MONTHLY_EXPENSE:,.0f}원')
print(f'충당 가능 기간: {months_covered:.1f}개월')
print()
if months_covered < 12:
    shortage = (12 - months_covered) * MONTHLY_EXPENSE
    print(f'⚠ 12개월치를 채우려면 {shortage:,.0f}원이 더 필요합니다.')
else:
    print(f'✅ 12개월치 이상 확보되어 있습니다.')

=== 생활비 안전도 ===
🟢 상태: 녹색 (안전)
현금성 자산: 35,000,000원
월 생활비:   2,500,000원
충당 가능 기간: 14.0개월

✅ 12개월치 이상 확보되어 있습니다.


## 4. 목표 비중 vs 현재 비중 비교

In [5]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'
from plotly.subplots import make_subplots

# 현재 비중
type_sum = df.groupby('asset_type')['current_value'].sum()
current = {
    'cash':   type_sum.get('cash',   0) / total,
    'bond':   (type_sum.get('bond',  0) + type_sum.get('tdf', 0)) / total,
    'equity': type_sum.get('equity', 0) / total,
    'income': type_sum.get('income', 0) / total,
}

labels  = ['현금성', '채권/TDF', '주식형', '리츠/인컴']
target  = [TARGET['target_cash'], TARGET['target_bond'],
           TARGET['target_equity'], TARGET['target_income']]
current_vals = [current['cash'], current['bond'],
                current['equity'], current['income']]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['현재 자산 배분', '목표 vs 현재 비중 비교'],
    specs=[[{'type': 'pie'}, {'type': 'bar'}]]
)

# 파이차트
fig.add_trace(go.Pie(
    labels=labels,
    values=[v * total for v in current_vals],
    hole=0.4,
    textinfo='label+percent',
    textfont_size=14,
), row=1, col=1)

# 막대차트 (목표 vs 현재)
fig.add_trace(go.Bar(
    name='목표', x=labels,
    y=[v * 100 for v in target],
    marker_color='#1F3864',
    text=[f'{v*100:.0f}%' for v in target],
    textposition='outside'
), row=1, col=2)

fig.add_trace(go.Bar(
    name='현재', x=labels,
    y=[v * 100 for v in current_vals],
    marker_color='#2E75B6',
    text=[f'{v*100:.1f}%' for v in current_vals],
    textposition='outside'
), row=1, col=2)

fig.update_layout(
    title_text=f'포트폴리오 현황  |  총 자산 {total:,.0f}원',
    title_font_size=18,
    height=450,
    showlegend=True,
    font=dict(size=14)
)
fig.update_yaxes(title_text='비중 (%)', row=1, col=2)
fig.show()

## 5. 리밸런싱 필요 여부 간단 체크

In [6]:
threshold = TARGET['rebalance_threshold']

type_targets = {
    'cash':   TARGET['target_cash'],
    'bond':   TARGET['target_bond'],
    'equity': TARGET['target_equity'],
    'income': TARGET['target_income'],
}

print('=== 리밸런싱 체크 ===')
needs_rebalance = False

for k, label in [('cash','현금성'),('bond','채권/TDF'),('equity','주식형'),('income','리츠/인컴')]:
    tgt = type_targets[k]
    cur = current[k]
    diff = cur - tgt
    status = ''
    if abs(diff) >= threshold:
        status = '  ← ⚠ 조정 필요'
        needs_rebalance = True
    print(f'{label:8s}: 목표 {tgt*100:.0f}%  현재 {cur*100:.1f}%  차이 {diff*100:+.1f}%p{status}')

print()
if needs_rebalance:
    print('일부 자산군이 목표 비중에서 벗어났습니다. 04_rebalance.ipynb에서 조정안을 확인하세요.')
else:
    print('모든 자산군이 목표 비중 범위 안에 있습니다. 현재 포트폴리오는 안정적입니다.')

=== 리밸런싱 체크 ===
현금성     : 목표 25%  현재 51.8%  차이 +26.8%p  ← ⚠ 조정 필요
채권/TDF  : 목표 25%  현재 46.1%  차이 +21.1%p  ← ⚠ 조정 필요
주식형     : 목표 35%  현재 1.5%  차이 -33.5%p  ← ⚠ 조정 필요
리츠/인컴   : 목표 15%  현재 0.6%  차이 -14.4%p  ← ⚠ 조정 필요

일부 자산군이 목표 비중에서 벗어났습니다. 04_rebalance.ipynb에서 조정안을 확인하세요.


## 6. 오늘 데이터를 DB에 저장

In [7]:
import sqlite3
from datetime import date

db_path = PROJECT_ROOT / 'portfolio.db'
today   = str(date.today())

conn = sqlite3.connect(db_path)
cur  = conn.cursor()

# 오늘 데이터가 이미 있으면 업데이트, 없으면 삽입
cur.execute('SELECT id FROM bucket_snapshots WHERE date = ?', (today,))
row = cur.fetchone()

if row:
    cur.execute('''
        UPDATE bucket_snapshots
        SET bucket1=?, bucket2=?, bucket3=?, total=?
        WHERE date=?
    ''', (b1, b2, b3, total, today))
    print(f'오늘({today}) 스냅샷 업데이트 완료')
else:
    cur.execute('''
        INSERT INTO bucket_snapshots (date, bucket1, bucket2, bucket3, total)
        VALUES (?, ?, ?, ?, ?)
    ''', (today, b1, b2, b3, total))
    print(f'오늘({today}) 스냅샷 저장 완료')

conn.commit()
conn.close()

print()
print(f'  버킷 1 (현금성):  {b1:>15,.0f}원')
print(f'  버킷 2 (채권):    {b2:>15,.0f}원')
print(f'  버킷 3 (성장):    {b3:>15,.0f}원')
print(f'  총 자산:          {total:>15,.0f}원')

오늘(2026-05-20) 스냅샷 저장 완료

  버킷 1 (현금성):       35,000,000원
  버킷 2 (채권):         31,130,000원
  버킷 3 (성장):          1,420,000원
  총 자산:               67,550,000원


---

## 완료!

자산 데이터가 로드되고 버킷이 분류되었습니다.

- **실제 자산 데이터로 교체**하려면: `assets_sample.csv` → `assets.csv`로 복사 후 수정
- **다음 단계**: `02_bucket_engine.ipynb` — 버킷 상세 분석